# Proyecto 1 — Ejercicio A: Precio de Vivienda

Notebook ordenado con la solución del **Ejercicio A**:

- Parte 1: Regresión Lineal Múltiple
  - Etapa 1.2: Solución computacional con dataset completo
- Parte 2: Regresión Logística para clasificar viviendas PREMIUM

In [1]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## Dataset

Columnas:

- `Area`: área en m²
- `Habitaciones`: número de habitaciones
- `Antiguedad`: antigüedad en años
- `Distancia`: distancia al centro en km
- `Precio`: precio de venta en USD

In [2]:
data = np.array([
    [95, 4, 13, 12.5, 171600],
    [123, 2, 14, 8.0, 216200],
    [118, 2, 22, 15.0, 196900],
    [96, 4, 8, 5.5, 190200],
    [182, 4, 19, 9.0, 305700],
    [86, 4, 5, 3.5, 180100],
    [63, 2, 7, 18.0, 133600],
    [193, 2, 3, 4.0, 320000],
    [155, 2, 29, 22.0, 242700],
    [128, 2, 3, 6.5, 216800],
    [195, 5, 18, 7.5, 357400],
    [115, 5, 22, 20.0, 219800],
    [105, 3, 10, 2.0, 198500],
    [140, 3, 6, 14.0, 241300]
], dtype=float)

columns = ['Area', 'Habitaciones', 'Antiguedad', 'Distancia', 'Precio']
df = pd.DataFrame(data, columns=columns)
df

,Area,Habitaciones,Antiguedad,Distancia,Precio
0,95.0000,4.0000,13.0000,12.5000,171600.0000
1,123.0000,2.0000,14.0000,8.0000,216200.0000
2,118.0000,2.0000,22.0000,15.0000,196900.0000
3,96.0000,4.0000,8.0000,5.5000,190200.0000
4,182.0000,4.0000,19.0000,9.0000,305700.0000
5,86.0000,4.0000,5.0000,3.5000,180100.0000
6,63.0000,2.0000,7.0000,18.0000,133600.0000
7,193.0000,2.0000,3.0000,4.0000,320000.0000
8,155.0000,2.0000,29.0000,22.0000,242700.0000
9,128.0000,2.0000,3.0000,6.5000,216800.0000


# Parte 1 — Regresión Lineal Múltiple

Modelo base:

$$
\hat{y}=w_0+w_1x_1+w_2x_2+w_3x_3+w_4x_4
$$

## Etapa 1.1 — Gradiente Descendiente a mano

Se usan solo los 3 primeros registros y una única característica: **Área**.

Valores iniciales:

$$
w_0=0, \quad w_1=1.0, \quad \alpha=0.1
$$

Se ejecutan exactamente 4 iteraciones.

In [3]:
x = np.array([0.95, 1.23, 1.18])
y_manual = np.array([1.71, 2.16, 1.96])

w0, w1 = 0.0, 1.0
lr = 0.1
n = len(x)

rows = []

for iteration in range(1, 5):
    y_pred = w0 + w1 * x
    error = y_pred - y_manual

    mse = np.mean(error ** 2)

    grad_w0 = (2 / n) * np.sum(error)
    grad_w1 = (2 / n) * np.sum(error * x)

    w0_new = w0 - lr * grad_w0
    w1_new = w1 - lr * grad_w1

    rows.append([
        iteration,
        w0,
        w1,
        mse,
        grad_w0,
        grad_w1,
        w0_new,
        w1_new
    ])

    w0, w1 = w0_new, w1_new

manual_results = pd.DataFrame(
    rows,
    columns=[
        'Iteracion',
        'w0 inicial',
        'w1 inicial',
        'MSE',
        'grad w0',
        'grad w1',
        'w0 nuevo',
        'w1 nuevo'
    ]
)

manual_results.round(4)

,Iteracion,w0 inicial,w1 inicial,MSE,grad w0,grad w1,w0 nuevo,w1 nuevo
0,1,0.0000,1.0000,0.6836,-1.6467,-1.8575,0.1647,1.1858
1,2,0.1647,1.1858,0.2069,-0.9012,-1.0171,0.2548,1.2875
2,3,0.2548,1.2875,0.0640,-0.4932,-0.5571,0.3041,1.3432
3,4,0.3041,1.3432,0.0211,-0.2697,-0.3052,0.3311,1.3737


In [4]:
print(f'Pesos finales despues de 4 iteraciones:')
print(f'w0 = {w0:.4f}')
print(f'w1 = {w1:.4f}')
print(f'Modelo final: y_hat = {w0:.4f} + {w1:.4f}x')

Pesos finales despues de 4 iteraciones:
w0 = 0.3311
w1 = 1.3737
Modelo final: y_hat = 0.3311 + 1.3737x


## Etapa 1.2 — Solución computacional

Se usa el dataset completo de 14 propiedades con 4 características.

Se construye:

$$
X \in \mathbb{R}^{14\times5}
$$

porque se agrega una columna de unos para el intercepto.

In [5]:
X_raw = df[['Area', 'Habitaciones', 'Antiguedad', 'Distancia']].values
y = df[['Precio']].values

print('Dimensiones:')
print('X_raw:', X_raw.shape)
print('y:', y.shape)

ranges = pd.DataFrame({
    'Minimo': df.min(),
    'Maximo': df.max()
})

ranges

Dimensiones:
X_raw: (14, 4)
y: (14, 1)


,Minimo,Maximo
Area,63.0000,195.0000
Habitaciones,2.0000,5.0000
Antiguedad,3.0000,29.0000
Distancia,2.0000,22.0000
Precio,133600.0000,357400.0000


In [6]:
# Normalizacion Z-score
mu = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)

X_norm = (X_raw - mu) / sigma

# Agregar columna de unos para w0
X = np.c_[np.ones(X_norm.shape[0]), X_norm]

print('Dimensiones de X con intercepto:', X.shape)

stats = pd.DataFrame({
    'Media': mu,
    'Desviacion estandar': sigma
}, index=['Area', 'Habitaciones', 'Antiguedad', 'Distancia'])

stats

Dimensiones de X con intercepto: (14, 5)


,Media,Desviacion estandar
Area,128.1429,39.1223
Habitaciones,3.1429,1.1249
Antiguedad,12.7857,7.8484
Distancia,10.5357,6.1947


In [7]:
# Ecuacion normal
w_normal = np.linalg.inv(X.T @ X) @ X.T @ y

coef_names = ['w0', 'w1_Area', 'w2_Habitaciones', 'w3_Antiguedad', 'w4_Distancia']

coef_normal_table = pd.DataFrame({
    'Coeficiente': coef_names,
    'Valor': w_normal.flatten()
})

print('Coeficientes por ecuacion normal:')
print(w_normal)
coef_normal_table.round(4)

Coeficientes por ecuacion normal:
[[227914.28571429]
 [ 57795.54784352]
 [ 10267.29157526]
 [ -3104.36433176]
 [ -2496.45220919]]


,Coeficiente,Valor
0,w0,227914.2857
1,w1_Area,57795.5478
2,w2_Habitaciones,10267.2916
3,w3_Antiguedad,-3104.3643
4,w4_Distancia,-2496.4522


In [8]:
# Gradiente Descendiente computacional
w_gd = np.zeros((X.shape[1], 1))
lr = 0.01
epochs = 5000
n = len(y)

mse_history = []

for epoch in range(1, epochs + 1):
    y_pred = X @ w_gd
    error = y_pred - y

    grad = (2 / n) * X.T @ error
    w_gd = w_gd - lr * grad

    if epoch % 500 == 0:
        mse = np.mean(error ** 2)
        mse_history.append([epoch, mse])
        print(f'Epoch {epoch}: MSE = {mse:.4f}')

mse_table = pd.DataFrame(mse_history, columns=['Epoca', 'MSE'])
mse_table.round(4)

Epoch 500: MSE = 54844266.0338
Epoch 1000: MSE = 54052809.9568
Epoch 1500: MSE = 54046790.4622
Epoch 2000: MSE = 54046744.6748
Epoch 2500: MSE = 54046744.3265


Epoch 3000: MSE = 54046744.3238
Epoch 3500: MSE = 54046744.3238
Epoch 4000: MSE = 54046744.3238
Epoch 4500: MSE = 54046744.3238
Epoch 5000: MSE = 54046744.3238


,Epoca,MSE
0,500,54844266.0338
1,1000,54052809.9568
2,1500,54046790.4622
3,2000,54046744.6748
4,2500,54046744.3265
5,3000,54046744.3238
6,3500,54046744.3238
7,4000,54046744.3238
8,4500,54046744.3238
9,5000,54046744.3238


In [9]:
coef_gd_table = pd.DataFrame({
    'Coeficiente': coef_names,
    'Valor': w_gd.flatten()
})

print('Coeficientes por Gradiente Descendiente:')
print(w_gd)
coef_gd_table.round(4)

Coeficientes por Gradiente Descendiente:
[[227914.28571429]
 [ 57795.54784336]
 [ 10267.29157516]
 [ -3104.3643314 ]
 [ -2496.45220953]]


,Coeficiente,Valor
0,w0,227914.2857
1,w1_Area,57795.5478
2,w2_Habitaciones,10267.2916
3,w3_Antiguedad,-3104.3643
4,w4_Distancia,-2496.4522


In [10]:
# Convertir el modelo normalizado a escala original
w0_norm = w_normal[0, 0]
w_features_norm = w_normal[1:, 0]

w_features_original = w_features_norm / sigma
w0_original = w0_norm - np.sum((w_features_norm * mu) / sigma)

original_scale_table = pd.DataFrame({
    'Coeficiente': ['w0_original', 'Area', 'Habitaciones', 'Antiguedad', 'Distancia'],
    'Valor': np.r_[w0_original, w_features_original]
})

print('Modelo en escala original:')
print('w0:', w0_original)
print('coeficientes:', w_features_original)
original_scale_table.round(4)

Modelo en escala original:
w0: 19224.422963815334
coeficientes: [1477.30605936 9127.63133804 -395.54246912 -403.00114494]


,Coeficiente,Valor
0,w0_original,19224.4230
1,Area,1477.3061
2,Habitaciones,9127.6313
3,Antiguedad,-395.5425
4,Distancia,-403.0011


In [11]:
# Prediccion: 120 m2, 3 habitaciones, 10 anios, 7 km
new_house = np.array([[120, 3, 10, 7]], dtype=float)

new_house_norm = (new_house - mu) / sigma
new_house_X = np.c_[np.ones(new_house_norm.shape[0]), new_house_norm]

prediction = new_house_X @ w_normal

print('Prediccion para casa de 120 m2, 3 habitaciones, 10 anios y 7 km:')
print(f'Precio predicho: ${prediction[0, 0]:,.2f} USD')

Prediccion para casa de 120 m2, 3 habitaciones, 10 anios y 7 km:
Precio predicho: $217,107.61 USD


# Parte 2 — Regresión Logística

Se convierte el precio en una variable binaria:

$$
y = \begin{cases}
1, & \text{PREMIUM: Precio} > 200000 \\
0, & \text{ESTÁNDAR: Precio} \leq 200000
\end{cases}
$$

Modelo logístico:

$$
\hat{p}=\sigma(w_0+w_1x_1+w_2x_2+w_3x_3+w_4x_4)
$$

In [12]:
y_bin = (y > 200000).astype(int)

class_0 = np.sum(y_bin == 0)
class_1 = np.sum(y_bin == 1)

print('Etapa 2.1 — Variable objetivo binaria')
print('Clase 0 ESTANDAR:', class_0)
print('Clase 1 PREMIUM:', class_1)

if class_0 == class_1:
    print('El dataset esta balanceado.')
else:
    print('El dataset no esta perfectamente balanceado, pero el desbalance es leve.')

class_table = pd.DataFrame({
    'Clase': [0, 1],
    'Categoria': ['ESTANDAR', 'PREMIUM'],
    'Cantidad': [class_0, class_1]
})

class_table

Etapa 2.1 — Variable objetivo binaria
Clase 0 ESTANDAR: 6
Clase 1 PREMIUM: 8
El dataset no esta perfectamente balanceado, pero el desbalance es leve.


,Clase,Categoria,Cantidad
0,0,ESTANDAR,6
1,1,PREMIUM,8


In [13]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


def binary_cross_entropy(y_true, y_prob):
    eps = 1e-12
    return -np.mean(
        y_true * np.log(y_prob + eps) +
        (1 - y_true) * np.log(1 - y_prob + eps)
    )


def train_logistic_regression(X_feature, y_true, lr=0.01, epochs=2000):
    n = len(y_true)

    X_design = np.c_[np.ones((X_feature.shape[0], 1)), X_feature]
    w = np.zeros((X_design.shape[1], 1))

    for epoch in range(1, epochs + 1):
        z = X_design @ w
        y_prob = sigmoid(z)

        grad = (1 / n) * X_design.T @ (y_prob - y_true)
        w = w - lr * grad

    y_prob = sigmoid(X_design @ w)
    y_pred = (y_prob >= 0.5).astype(int)
    accuracy = np.mean(y_pred == y_true)
    loss = binary_cross_entropy(y_true, y_prob)

    return w, y_prob, y_pred, accuracy, loss


def confusion_matrix_2x2(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=int)

    for real, pred in zip(y_true.flatten(), y_pred.flatten()):
        cm[int(real), int(pred)] += 1

    return cm

## Etapa 2.2 — Modelos logísticos univariados

Se entrena un modelo por cada variable individual usando:

$$
lr=0.01, \quad epochs=2000
$$

In [14]:
feature_names = ['Area', 'Habitaciones', 'Antiguedad', 'Distancia']

univariate_results = []

for i, name in enumerate(feature_names):
    X_uni = X_norm[:, [i]]

    w_uni, prob_uni, pred_uni, acc_uni, loss_uni = train_logistic_regression(
        X_uni,
        y_bin,
        lr=0.01,
        epochs=2000
    )

    univariate_results.append([
        name,
        w_uni[0, 0],
        w_uni[1, 0],
        acc_uni,
        loss_uni
    ])

    print(f'Modelo usando solo {name}')
    print('w0:', w_uni[0, 0])
    print('w_feature:', w_uni[1, 0])
    print('Accuracy:', acc_uni)
    print()

univariate_table = pd.DataFrame(
    univariate_results,
    columns=['Modelo', 'w0', 'w_feature', 'Accuracy', 'BCE']
)

univariate_table.round(4)

Modelo usando solo Area
w0: 0.6248688984168306
w_feature: 2.4086529708082978
Accuracy: 0.8571428571428571



Modelo usando solo Habitaciones
w0: 0.28567109338728264
w_feature: -0.03670421241339989
Accuracy: 0.5714285714285714



Modelo usando solo Antiguedad
w0: 0.30131569947504877
w_feature: 0.4509165600600346
Accuracy: 0.5

Modelo usando solo Distancia
w0: 0.2931871589174048
w_feature: 0.32087815979064543
Accuracy: 0.6428571428571429



,Modelo,w0,w_feature,Accuracy,BCE
0,Area,0.6249,2.4087,0.8571,0.2723
1,Habitaciones,0.2857,-0.0367,0.5714,0.6827
2,Antiguedad,0.3013,0.4509,0.5000,0.6591
3,Distancia,0.2932,0.3209,0.6429,0.6705


## Etapa 2.3 — Modelo logístico completo

Se entrena con las 4 variables usando:

$$
lr=0.01, \quad epochs=3000
$$

In [15]:
w_full, prob_full, pred_full, acc_full, loss_full = train_logistic_regression(
    X_norm,
    y_bin,
    lr=0.01,
    epochs=3000
)

full_coef_table = pd.DataFrame({
    'Coeficiente': ['w0', 'w1_Area', 'w2_Habitaciones', 'w3_Antiguedad', 'w4_Distancia'],
    'Valor': w_full.flatten()
})

print('Pesos modelo completo:')
print('w0:', w_full[0, 0])
print('w1 Area:', w_full[1, 0])
print('w2 Habitaciones:', w_full[2, 0])
print('w3 Antiguedad:', w_full[3, 0])
print('w4 Distancia:', w_full[4, 0])
print()
print('Accuracy modelo completo:', acc_full)

full_coef_table.round(4)

Pesos modelo completo:
w0: 0.7637474139476369
w1 Area: 2.814674981559574
w2 Habitaciones: 0.014055311186859933
w3 Antiguedad: -0.36895532572504286
w4 Distancia: 0.6973283117285275

Accuracy modelo completo: 0.9285714285714286


,Coeficiente,Valor
0,w0,0.7637
1,w1_Area,2.8147
2,w2_Habitaciones,0.0141
3,w3_Antiguedad,-0.3690
4,w4_Distancia,0.6973


In [16]:
comparison_rows = [[row[0], row[3]] for row in univariate_results]
comparison_rows.append(['Completo', acc_full])

accuracy_comparison = pd.DataFrame(comparison_rows, columns=['Modelo', 'Accuracy'])
accuracy_comparison['Accuracy (%)'] = accuracy_comparison['Accuracy'] * 100

accuracy_comparison.round(4)

,Modelo,Accuracy,Accuracy (%)
0,Area,0.8571,85.7143
1,Habitaciones,0.5714,57.1429
2,Antiguedad,0.5000,50.0000
3,Distancia,0.6429,64.2857
4,Completo,0.9286,92.8571


In [17]:
cm = confusion_matrix_2x2(y_bin, pred_full)

print('Matriz de confusion:')
print(cm)

confusion_table = pd.DataFrame(
    cm,
    index=['Real 0', 'Real 1'],
    columns=['Pred 0', 'Pred 1']
)

confusion_table

Matriz de confusion:
[[5 1]
 [0 8]]


,Pred 0,Pred 1
Real 0,5,1
Real 1,0,8


In [18]:
# Prediccion nueva propiedad: 175 m2, 4 habitaciones, 8 anios, 6 km
new_house_log = np.array([[175, 4, 8, 6]], dtype=float)

new_house_log_norm = (new_house_log - mu) / sigma
new_house_log_X = np.c_[np.ones((new_house_log_norm.shape[0], 1)), new_house_log_norm]

prob_premium = sigmoid(new_house_log_X @ w_full)
class_pred = (prob_premium >= 0.5).astype(int)

print('Prediccion nueva propiedad:')
print('Area: 175 m2')
print('Habitaciones: 4')
print('Antiguedad: 8 anios')
print('Distancia: 6 km')
print('P(PREMIUM):', prob_premium[0, 0])
print('Clase predicha:', class_pred[0, 0])

Prediccion nueva propiedad:
Area: 175 m2
Habitaciones: 4
Antiguedad: 8 anios
Distancia: 6 km
P(PREMIUM): 0.9793670979114337
Clase predicha: 1


## Resumen final

- La regresión lineal predijo aproximadamente **217,108 USD** para la vivienda de 120 m², 3 habitaciones, 10 años y 7 km.
- En regresión logística, el mejor modelo fue el multivariable con accuracy de **92.86%**.
- La vivienda de 175 m², 4 habitaciones, 8 años y 6 km fue clasificada como **PREMIUM** con probabilidad aproximada de **97.94%**.